In [ ]:
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:
### Curvature Z-score
# Clip raw DNA curvature values, perform Z-score normalization.
# Mean and standard deviation are calculated only from training chromosomes to avoid data leakage.
# Normalized outputs are saved per chromosome, and normalization statistics are stored in json file.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

CURV_CSV = f"{root_dir}/DNAcurve/bin_dna_curvature.csv"
out_dir = f"{root_dir}/DNAcurve/norm/"
os.makedirs(out_dir, exist_ok=True)

train_chrs   = [f"chr{i}" for i in range(1,13)]
val_chrs     = [f"chr{i}" for i in range(13,20)]
test_chrs    = ["chr20","chr21","chr22","chrX"]

df = pd.read_csv(CURV_CSV)
coord_cols = ["chr", "start", "end"]
signal_cols = ["curv_mean"]
coord_key = "chr"

df[signal_cols] = df[signal_cols].apply(pd.to_numeric, errors="coerce")
df = df.sort_values(by=[coord_key, "start"], ascending=True).reset_index(drop=True)

df_proc = df.copy()
df_proc[signal_cols] = df_proc[signal_cols].clip(lower=0.0, upper=0.5)

# Calculate statistics using only non-NaN values from the training set
train_subset = df_proc[df_proc[coord_key].isin(train_chrs)]
stats = {}
for c in signal_cols:
    vals = train_subset[c].dropna()
    stats[c] = {"mu": float(vals.mean()), "sd": float(vals.std())}

# Z-score, keep NaN
df_norm = df_proc.copy()
for c in signal_cols:
    mu = stats[c]["mu"]
    sd = stats[c]["sd"]
    mask_valid = ~df_norm[c].isna()
    if sd < 1e-8:
        df_norm.loc[mask_valid, c] = 0.0
    else:
        df_norm.loc[mask_valid, c] = (df_norm.loc[mask_valid, c] - mu) / sd

df_output = pd.concat([df[coord_cols], df_norm[signal_cols]], axis=1)

# Output by chromosome
def save_single_chrom_tsv(chr_name):
    sub = df_output[df_output[coord_key]==chr_name].reset_index(drop=True)
    starts = sub["start"].values
    diffs = np.diff(starts)
    if not np.all(diffs == 1000):
        gap_pos = np.where(diffs != 1000)[0]
        print(f"WARNING: {chr_name} has a 1kb breakpoint, the first 5 breakpoint indices：{gap_pos[:5]}")
    tsv_path = os.path.join(out_dir, f"{chr_name}_curv1kb.tsv")
    sub.to_csv(tsv_path, sep="\t", index=False)
    print(f"Saved {chr_name} → {tsv_path} | bin_count={sub.shape[0]}")

all_chroms = train_chrs + val_chrs + test_chrs
for chrom in all_chroms:
    save_single_chrom_tsv(chrom)

with open(os.path.join(out_dir, "curv_norm_stats.json"), "w") as f:
    json.dump(stats, f, indent=2)

print("\n==== Curvature Z-score statistical parameters of the training set ====")
for k, v in stats.items():
    print(f"{k:20s}: mu={v['mu']:.6f}, sd={v['sd']:.6f}")

In [ ]:
### Train set DNA curvature

In [ ]:

import pandas as pd
import numpy as np
import os

SEED_LIST = [42, 43, 44, 45, 46]
for SEED in SEED_LIST:
    print(f"=========== SEED:{SEED} ============")
    CURV_ROOT = f"{root_dir}/DNAcurve/norm/"
    COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/train/hic_train_coords_{display_reso}kb_seed{SEED}.csv"
    OUT_NPY = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/train/curv{cell}_train_feat100_seed{SEED}.npy"
    OUT_MASK = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/train/curv{cell}_train_mask100_seed{SEED}.npy"
    
    signal_names = ["curv_mean"]
    n_seg_total = 100
    seg_length = 1000
    
    hic_df = pd.read_csv(COORD_CSV)
    all_features = []
    all_masks = []
    
    # train set chr1‑chr12
    chrs_order = [f"chr{i}" for i in range(1,13)]
    
    for chrom in chrs_order:
        group_df = hic_df[hic_df["chr"] == chrom]
        print(f"Processing {chrom}, total bins: {len(group_df)}")
        curv_tsv_path = os.path.join(CURV_ROOT, f"{chrom}_curv1kb.tsv")
    
        if not os.path.exists(curv_tsv_path):
            print(f"!!! {curv_tsv_path} not found, skip {chrom}")
            continue
    
        curv_df = pd.read_csv(curv_tsv_path, sep="\t")
        curv_dict = {}
        for _, row in curv_df.iterrows():
            s = int(row["start"])
            feats = row[signal_names].values.astype(np.float32)
            curv_dict[s] = feats
    
        for _, bin_row in group_df.iterrows():
            bin_start = int(bin_row["start"])
            bin_end = int(bin_row["end"])
            center = bin_start + (bin_end - bin_start) / 2
            win_left = int(center - 50000)
    
            raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
            for seg_idx in range(n_seg_total):
                seg_l = win_left + seg_idx * seg_length
                curv_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
                if curv_start_candidate in curv_dict:
                    raw_100[seg_idx] = curv_dict[curv_start_candidate]
    
            out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
            out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)
    
            all_features.append(out_feat)
            all_masks.append(out_mask)
    
    feat_arr = np.stack(all_features, axis=0)
    mask_arr = np.stack(all_masks, axis=0)
    
    np.save(OUT_NPY, feat_arr)
    np.save(OUT_MASK, mask_arr)
    
    print(f"\nDone! hic csv total rows: {len(hic_df)}")
    print(f"curv feature shape: {feat_arr.shape}")   # (N,100,1)
    print(f"curv mask shape: {mask_arr.shape}")     # (N,100)

In [ ]:
### Validation set DNA curvature

In [ ]:

import pandas as pd
import numpy as np
import os

CURV_ROOT = f"{root_dir}/DNAcurve/norm/"   
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/hic_val_coords13-19_{display_reso}kb.csv"
OUT_NPY = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/curv{cell}_validation_feat100.npy"
OUT_MASK = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/validation/curv{cell}_validation_mask100.npy"

signal_names = ["curv_mean"]
n_seg_total = 100    
seg_length = 1000

hic_df = pd.read_csv(COORD_CSV)
all_features = []
all_masks = []

chrs_order = [f"chr{i}" for i in range(13,20)]

for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    curv_tsv_path = os.path.join(CURV_ROOT, f"{chrom}_curv1kb.tsv")

    if not os.path.exists(curv_tsv_path):
        print(f"!!! {curv_tsv_path} not found, skip {chrom}")
        continue

    curv_df = pd.read_csv(curv_tsv_path, sep="\t")
    curv_dict = {}
    for _, row in curv_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        curv_dict[s] = feats

    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        center = bin_start + (bin_end - bin_start) / 2
        win_left = int(center - 50000)

        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            curv_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if curv_start_candidate in curv_dict:
                raw_100[seg_idx] = curv_dict[curv_start_candidate]

        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)

        all_features.append(out_feat)
        all_masks.append(out_mask)

feat_arr = np.stack(all_features, axis=0)
mask_arr = np.stack(all_masks, axis=0)

np.save(OUT_NPY, feat_arr)
np.save(OUT_MASK, mask_arr)

print(f"\nDone! hic csv total rows: {len(hic_df)}")
print(f"curv feature shape: {feat_arr.shape}")   # (N,100,1)
print(f"curv mask shape: {mask_arr.shape}")     # (N,100)


In [ ]:
### Test set

In [ ]:

import pandas as pd
import numpy as np
import os

CURV_ROOT = f"{root_dir}/DNAcurve/norm/" 
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/hic_predict_coords20-X_{display_reso}kb.csv"
OUT_NPY = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_feat100.npy"
OUT_MASK = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_mask100.npy"

signal_names = ["curv_mean"]
n_seg_total = 100
seg_length = 1000

hic_df = pd.read_csv(COORD_CSV)
all_features = []
all_masks = []

chrs_order = [f"chr{i}" for i in range(20,23)]+["chrX"]

for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    curv_tsv_path = os.path.join(CURV_ROOT, f"{chrom}_curv1kb.tsv")

    if not os.path.exists(curv_tsv_path):
        print(f"!!! {curv_tsv_path} not found, skip {chrom}")
        continue

    curv_df = pd.read_csv(curv_tsv_path, sep="\t")
    curv_dict = {}
    for _, row in curv_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        curv_dict[s] = feats

    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        center = bin_start + (bin_end - bin_start) / 2
        win_left = int(center - 50000)

        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            curv_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if curv_start_candidate in curv_dict:
                raw_100[seg_idx] = curv_dict[curv_start_candidate]

        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)

        all_features.append(out_feat)
        all_masks.append(out_mask)

feat_arr = np.stack(all_features, axis=0)
mask_arr = np.stack(all_masks, axis=0)

np.save(OUT_NPY, feat_arr)
np.save(OUT_MASK, mask_arr)

print(f"\nDone! hic csv total rows: {len(hic_df)}")
print(f"curv feature shape: {feat_arr.shape}")   # (N,100,1)
print(f"curv mask shape: {mask_arr.shape}")     # (N,100)